# Problem set 6: Solving the Solow model

In [1]:
import numpy as np
from scipy import linalg
from scipy import optimize
import sympy as sm

# Problem: Solve the Solow model

## Introduction

Consider the **standard Solow-model** where:

1. $K_t$ is capital2
2. $L_t$ is labor (growing with a constant rate of $n$)
3. $A_t$ is technology (growing with a constant rate of $g$)
4. $Y_t = F(K_t,A_tL_t)$ is GDP

**Saving** is a constant fraction of GDP

$$ 
S_t = sY_t,\,s\in(0,1)
$$

such that **capital accumulates** according to

$$
K_{t+1}=S_{t}+(1-\delta)K_{t}=sF(K_{t},A_{t}L_{t})+(1-\delta)K_{t}, \delta \in (0,1)
$$

The **production function** has **constant-return to scale** such that

$$
\frac{Y_{t}}{A_{t}L_{t}}=\frac{F(K_{t},A_{t}L_{t})}{A_{t}L_{t}}=F(\tilde{k}_{t},1)\equiv f(\tilde{k}_{t})
$$

where $\tilde{k}_t = \frac{K_t}{A_{t}L_{t}}$ is the technology adjusted capital-labor ratio.

The **transition equation** then becomes

$$
\tilde{k}_{t+1}= \frac{1}{(1+n)(1+g)}[sf(\tilde{k}_{t})+(1-\delta)\tilde{k}_{t}]
$$

If the **production function** is **Cobb-Douglas** then

$$
F(K_{t},A_{t}L_{t})=K_{t}^{\alpha}(A_{t}L_{t})^{1-\alpha}\Rightarrow f(\tilde{k}_{t})=\tilde{k}_{t}^{\alpha}
$$

If it is **CES** (with $\beta < 1, \beta \neq 0$) then

$$
F(K_{t},A_{t}L_{t})=(\alpha K_{t}^{\beta}+(1-\alpha)(A_{t}L_{t})^{\beta})^{\frac{1}{\beta}}\Rightarrow f(\tilde{k}_{t})=(\alpha\tilde{k}_{t}^{\beta}+(1-\alpha))^{\frac{1}{\beta}}
$$

## Steady state

Assume the production function is **Cobb-Douglas**.

**Question A:** Use **sympy** to find an analytical expression for the steady state, i.e. solve

$$
\tilde{k}^{\ast}= \frac{1}{(1+n)(1+g)}[sf(\tilde{k}^{\ast})+(1-\delta)\tilde{k}^{\ast}]
$$

In [2]:
k = sm.symbols('k')
alpha = sm.symbols('alpha')
delta = sm.symbols('delta')
s = sm.symbols('s')
g = sm.symbols('g')
n = sm.symbols('n')

In [3]:
# write your code here

f = k**alpha #Defines the production function where f represents output, k represents capital, and α represents the share of capital in output production.
ss = sm.Eq(k,(s*f+(1-delta)*k)/((1+n)*(1+g))) #Sets up the steady-state equation, which equates investment (savings times output, s×f) plus the depreciation of capital ((1−δ)×k) to the increase in capital due to population growth (n×k) and technological progress (g×k).
print(ss)
kss = sm.solve(ss,k)[0] #Solves the steady-state equation for the steady-state value of capital (k_ss) using SymPy's solve function and takes the first solution (assuming there's only one solution), storing it in the variable kss.
kss
# The result stored in 'kss' will be the steady state value of capital.

Eq(k, (k*(1 - delta) + k**alpha*s)/((g + 1)*(n + 1)))


((delta + g*n + g + n)/s)**(1/(alpha - 1))

**Answer:** see A7.py

**Question B:** Turn your solution into a Python function called as `ss_func(s,g,n,delta,alpha)`. 

In [8]:

#This code defines a Python function ss_func that computes the steady-state value of capital (k_ss) based on the parameters of the Solow growth model.

ss_func = sm.lambdify((s,g,n,delta,alpha),kss) #This line creates a callable function ss_func that takes five arguments (s, g, n, delta, alpha) corresponding to the parameters of the Solow growth model, and returns the steady-state value of capital (kss). sm.lambdify is a function in SymPy that creates a function from a SymPy expression, allowing it to be evaluated numerically.
#I.e. we use the variable 'kss' that we created before with SymPy and create a function that allows us to evaluate it numerically.

# Evaluate function
ss_func(0.2,0.02,0.01,0.1,1/3) #This line calls the ss_func function with specific values for the parameters (s, g, n, delta, alpha). 
#In this case, it's evaluating the steady-state value of capital for the following parameter values:

    #s=0.2 (the savings rate)

    #g=0.02 (the technological growth rate)

    #n=0.01 (the population growth rate)

    #δ=0.1 (the depreciation rate)

    #α= 1/3 (the share of capital in output production)"""

1.903831539231319

**Answer:** A8.py

**Question C**: Find the steady state numerically using root-finding with `optimize.root_scalar`.

In [9]:
s = 0.2
g = 0.02
n = 0.01
alpha = 1/3
delta = 0.1


f = lambda k: k**alpha #It defines the production function f(k)=k^α using a lambda function.
obj_kss = lambda kss: kss - (s*f(kss) + (1-delta)*kss)/((1+g)*(1+n)) #It defines an objective function obj_kss that represents the difference between the left-hand side and right-hand side of the steady-state equation. This objective function calculates the deviation from steady state for a given k_ss.
result = optimize.root_scalar(obj_kss,bracket=[0.1,100],method='brentq') #It uses the root_scalar function from the optimize module (presumably from the SciPy library) to find the root of the obj_kss function, which corresponds to the steady-state value of capital (k_ss). The bracket parameter specifies an initial interval within which the root is searched, and the method='brentq' specifies the Brent's method for root finding.

print('the steady state for k is',result.root) #it prints the steady-state value of capital (k_ss) found by the optimization routine.

the steady state for k is 1.9038315392313185


**Answer:** A9.py

**Question D:** Now assume the production function is CES. Find the steady state for $k$ for the various values of $\beta$ shown below.

In [10]:
betas = [-0.5,-0.25,-0.1,-0.05,0.05,0.1,0.25,0.5] #Defines a list of beta values over which the loop will iterate.


for beta in betas: #Initiates a loop that iterates over each beta value in the betas list.
    f = lambda k: (alpha*k**beta + (1-alpha))**(1/beta) #A production function f using a lambda function. The production function is based on a Cobb-Douglas type of function
    obj_kss = lambda kss: kss - (s*f(kss) + (1-delta)*kss)/((1+g)*(1+n)) #An objective function obj_kss that represents the difference between the left-hand side and right-hand side of the steady-state equation, similar to the previous example.
    result = optimize.root_scalar(obj_kss,bracket=[0.1,100],method='brentq') #It then uses numerical optimization (root-finding) to find the steady-state value of capital (k_ss) for the given beta value using the root_scalar function from the optimize module
    print(f'for beta = {beta:.3f} the steady state for k is',result.root) #it prints the steady-state value of capital (k_ss) for the current beta value.

for beta = -0.500 the steady state for k is 1.8471297000972984
for beta = -0.250 the steady state for k is 1.873383262758588
for beta = -0.100 the steady state for k is 1.8910856397655083
for beta = -0.050 the steady state for k is 1.8973581025712736
for beta = 0.050 the steady state for k is 1.9105159729244352
for beta = 0.100 the steady state for k is 1.917422132817728
for beta = 0.250 the steady state for k is 1.9395902733676993
for beta = 0.500 the steady state for k is 1.9822334997701418


**Answer:** A10.py   

## **Our addition to the Solowmodel**

Add human capital to the model

## **Simulation**

**Plot and check with the analytical solution**

Does it converge to the analytical SS?

**Alternative scenarios**

Make parameter changes